# Stack & Queue

*LIFO · FIFO · Monotonic Stack · Real-World*



---
## 🧠 Mental Model: Stacks & Queues

> **Stack = last-in-first-out (LIFO). Queue = first-in-first-out (FIFO).**  
> Think of a stack as a pile of plates; a queue as a line at a coffee shop. The insertion and removal points determine the access pattern and the right data structure.

### WHY — Why do they exist?
They constrain access to data in ways that match specific computation patterns:
- **Stack**: processing things in reverse order of arrival (function calls, undo, DFS)
- **Queue**: processing things in arrival order (BFS, task scheduling, buffering)

### WHAT & HOW — Implementations

| Structure | Insert | Remove | Python impl | Time |
|-----------|--------|--------|-------------|------|
| Stack | push (top) | pop (top) | `list.append` / `.pop()` | O(1) amortised |
| Queue | enqueue (rear) | dequeue (front) | `collections.deque` | O(1) both ends |
| Deque | both ends | both ends | `collections.deque` | O(1) both ends |
| Thread-safe Queue | put | get | `queue.Queue` | O(1) + lock overhead |

**Critical**: `list.pop(0)` for dequeue is **O(n)** — never use it. Always use `collections.deque`.

### HOW — Monotonic Stack (advanced pattern)

A stack where elements are always kept in sorted order (increasing or decreasing). Used to find the "next greater/smaller element" in O(n):

```python
def next_greater(nums):
    result = [-1] * len(nums)
    stack = []              # stores indices
    for i, n in enumerate(nums):
        while stack and nums[stack[-1]] < n:
            result[stack.pop()] = n    # n is the NGE for the popped element
        stack.append(i)
    return result
```

### WHEN — Use cases

| Problem | Structure | Why |
|---------|-----------|-----|
| Balanced brackets `(){[]}` | Stack | Match pairs in reverse arrival order |
| DFS traversal | Stack | Process children before returning |
| Undo/Redo history | Stack | Reverse chronological order |
| BFS / level-order traversal | Queue | Process level-by-level in order |
| Task scheduling (FIFO) | Queue | Fair processing order |
| Sliding window maximum | Deque (monotonic) | O(n) max over each window |
| Next greater element | Monotonic stack | O(n) instead of O(n²) |
| Browser forward/back | Two stacks | Back stack + forward stack |

**Gotchas:**
1. `list.pop(0)` is O(n) — use `deque.popleft()` for O(1) dequeue
2. `list.append/pop()` is O(1) amortised — fine for stack use
3. `queue.Queue` is thread-safe but slower than `deque` — only use for multi-threading
4. Monotonic stacks process in one pass (O(n)) — recognise the pattern when you need "next greater/smaller"

```
Complexity:
  Stack push/pop:    O(1) amortised (list) or O(1) (deque)
  Queue enqueue:     O(1) (deque)
  Queue dequeue:     O(1) (deque) — O(n) with list.pop(0)!
  Space:             O(n)
```


Stack (LIFO): push/pop from THE SAME END.
  Uses: DFS, undo/redo, balanced brackets, call stack simulation.
Queue (FIFO): enqueue at one end, dequeue from the other.
  Uses: BFS, task scheduling, rate limiting, producer-consumer.

> ⚠️ **GOTCHA 1: list.pop(0) for queue dequeue is O(n) — always use collections.deque.**
> ⚠️ **GOTCHA 2: Python's list.append/pop() is O(1) amortised for STACK use.**
> ⚠️ **GOTCHA 3: queue.Queue (thread-safe) vs deque (not thread-safe but faster).**
> ⚠️ **GOTCHA 4: Monotonic stack — a stack whose elements are always ordered;**
  used for "next greater element", histogram area, etc.

In [ ]:
def notebook_stack_queue_gotchas() -> None:

## §3 · Stack & Queue Gotchas

In [ ]:
# ── §3.1  Balanced brackets (classic stack application) ───────────────
    def is_balanced(s):
        stack, pairs = [], {")":"(", "]":"[", "}":"{"}
        for ch in s:
            if ch in "([{":
                stack.append(ch)
            elif ch in ")]}":
                if not stack or stack[-1] != pairs[ch]:
                    return False
                stack.pop()
        return len(stack) == 0

    assert is_balanced("({[]})")
    assert not is_balanced("({[}])")
    assert is_balanced("")               # empty string is balanced
    assert not is_balanced("((")         # unclosed
    print("Balanced brackets: ✓")

    # ── §3.2  GOTCHA: list.pop(0) is O(n) ────────────────────────────────
    lst = list(range(5000))
    t0 = time.perf_counter()
    while lst: lst.pop(0)             # O(n) per pop  = O(n²) total
    t_list = time.perf_counter() - t0

    dq = deque(range(5000))
    t0 = time.perf_counter()
    while dq: dq.popleft()            # O(1) per pop  = O(n) total
    t_deque = time.perf_counter() - t0

    print(f"\nlist.pop(0) 5000×:    {t_list*1000:.2f}ms  (O(n²) total)")
    print(f"deque.popleft 5000×:  {t_deque*1000:.2f}ms  (O(n) total)")

    # ── §3.3  Monotonic stack — "next greater element" ───────────────────
    #
    # GOTCHA: people reach for O(n²) nested loops; monotonic stack is O(n).
    # Mental model: keep a stack of "candidates waiting for their answer".
    # When a new element is larger, it answers all smaller elements below it.

    def next_greater(nums):
        """For each element, find the next element to its right that is greater.
           Returns -1 if none exists.  O(n) time and space."""
        result = [-1] * len(nums)
        stack = []                         # stack of indices
        for i, n in enumerate(nums):
            while stack and nums[stack[-1]] < n:
                idx = stack.pop()
                result[idx] = n            # n is the next greater for idx
            stack.append(i)
        return result

    assert next_greater([2, 1, 2, 4, 3]) == [4, 2, 4, -1, -1]
    print("Monotonic stack next_greater([2,1,2,4,3]):", next_greater([2,1,2,4,3]))

    # ── §3.4  Largest rectangle in histogram ─────────────────────────────
    #
    # Classic stack problem — O(n) using monotonic increasing stack.
    def largest_rectangle(heights):
        stack, max_area = [], 0
        for i, h in enumerate(heights + [0]):   # append sentinel 0 to flush
            while stack and heights[stack[-1]] > h:
                height = heights[stack.pop()]
                width  = i if not stack else i - stack[-1] - 1
                max_area = max(max_area, height * width)
            stack.append(i)
        return max_area

    assert largest_rectangle([2,1,5,6,2,3]) == 10
    print("Largest rectangle in histogram:", largest_rectangle([2,1,5,6,2,3]))

A heap is a COMPLETE binary tree satisfying the heap property:
  Min-heap: parent ≤ both children (root = minimum element)
  Max-heap: parent ≥ both children (root = maximum element)
Python's heapq is a MIN-heap ONLY.  For max-heap: negate values.

O(1) peek at minimum (heap[0])
O(log n) push and pop
O(n) heapify (build from list — NOT O(n log n)!)

> ⚠️ **GOTCHA 1: heapq is NOT thread-safe; use queue.PriorityQueue for that.**
> ⚠️ **GOTCHA 2: heapq.nlargest(k, it) is O(n log k), not O(n log n) — use it!**
> ⚠️ **GOTCHA 3: Heap elements must be comparable; for custom objects use a tuple**
  (priority, tiebreaker, item) to avoid TypeError on equal priorities.
> ⚠️ **GOTCHA 4: Modifying an element already in the heap corrupts the heap property.**
  Use the "lazy deletion" pattern instead.

In [ ]:
def notebook_heap_gotchas() -> None:

## §5 · Heap / Priority Queue Gotchas

In [ ]:
import heapq

    # ── §5.1  Max-heap via negation ───────────────────────────────────────
    nums = [3, 1, 4, 1, 5, 9, 2, 6]
    max_heap = [-x for x in nums]
    heapq.heapify(max_heap)
    max_val = -heapq.heappop(max_heap)
    print(f"Max-heap via negation: max of {nums} = {max_val}")   # 9

    # ── §5.2  GOTCHA: equal priorities need tiebreaker ────────────────────
    from dataclasses import dataclass, field

    @dataclass(order=False)
    class Task:
        name: str
        priority: int

    # This would raise TypeError if two tasks have equal priority:
    # heapq.heappush(h, Task("a",1)); heapq.heappush(h, Task("b",1))
    # Fix: wrap in a tuple (priority, counter, task)

    counter = 0
    heap = []
    for t in [Task("low",3), Task("high",1), Task("med",2), Task("also-high",1)]:
        heapq.heappush(heap, (t.priority, counter, t))   # counter breaks ties
        counter += 1

    ordered = []
    while heap:
        _, _, task = heapq.heappop(heap)
        ordered.append(task.name)
    print(f"Priority order (ties by insertion): {ordered}")

    # ── §5.3  GOTCHA: heapify is O(n), not O(n log n) ────────────────────
    #
    # Floyd's algorithm: start from the last non-leaf and sift down.
    # Each sift-down is O(log k) for a subtree of size k.
    # Total: Σ O(h_i) for all nodes = O(n) (most nodes have small subtrees).

    big = list(range(100_000, 0, -1))   # reversed = worst case for sorting
    t0 = time.perf_counter()
    heapq.heapify(big)
    print(f"\nheapify(100k elements): {(time.perf_counter()-t0)*1000:.2f}ms — O(n)")

    # ── §5.4  Top-K pattern — O(n log k), better than sort O(n log n) ─────
    import heapq
    data = list(range(1, 1_000_001))   # 1M elements
    t0 = time.perf_counter()
    top_k = heapq.nlargest(10, data)
    t_heap = time.perf_counter() - t0

    t0 = time.perf_counter()
    top_k_sort = sorted(data, reverse=True)[:10]
    t_sort = time.perf_counter() - t0

    assert top_k == top_k_sort
    print(f"\nTop-10 from 1M elements:")
    print(f"  heapq.nlargest: {t_heap*1000:.2f}ms  (O(n log k))")
    print(f"  sort+slice:     {t_sort*1000:.2f}ms  (O(n log n))")
    print(f"  nlargest is {t_sort/t_heap:.1f}× faster for small k")

    # ── §5.5  Lazy deletion pattern ───────────────────────────────────────
    #
    # GOTCHA: you can't remove/update an arbitrary element from a heap.
    # Pattern: mark it as "deleted" in a set; skip when popping.

    heap2 = []
    deleted = set()
    uid = 0

    def push(priority, item):
        nonlocal uid
        entry = (priority, uid, item)
        heapq.heappush(heap2, entry)
        uid += 1
        return entry

    def remove(entry):
        deleted.add(entry)   # mark as deleted

    def pop():
        while heap2:
            entry = heapq.heappop(heap2)
            if entry not in deleted:
                return entry[2]   # return item
        return None

    e1 = push(3, "task-C"); push(1, "task-A"); push(2, "task-B")
    remove(e1)   # "cancel" task-C without heap restructure
    results = [pop() for _ in range(2)]
    print(f"\nLazy deletion: {results}")   # ['task-A', 'task-B']


def run_data_structures_notebook() -> None:
    notebook_linked_list_gotchas()
    notebook_lru_gotchas()
    notebook_stack_queue_gotchas()
    notebook_bst_gotchas()
    notebook_heap_gotchas()
    print("\n" + "═"*64)
    print("  DATA STRUCTURES NOTEBOOK COMPLETE")
    print("═"*64)


if __name__ == "__main__":
    if hasattr(sys.stdout, "reconfigure"):
        sys.stdout.reconfigure(encoding="utf-8")
    main()
    run_data_structures_notebook()

---
## SEPARATE CHAINING: slot -> list of (key, value) pairs. Simple and robust.


In [ ]:
class ChainingHashMap:
    def __init__(self, capacity: int = 8):
        self._cap = capacity
        self._size = 0
        self._buckets: list[list] = [[] for _ in range(capacity)]

    def _index(self, key) -> int:
        return hash(key) % self._cap        # fold the hash into [0, cap)

    def _resize(self, new_cap: int) -> None:
        old = [pair for bucket in self._buckets for pair in bucket]
        self._cap = new_cap
        self._buckets = [[] for _ in range(new_cap)]
        self._size = 0
        for k, v in old:
            self.put(k, v)                  # re-hash every entry into the bigger table

    def put(self, key, value) -> None:
        bucket = self._buckets[self._index(key)]
        for i, (k, _) in enumerate(bucket):
            if k == key:                    # update existing key
                bucket[i] = (key, value)
                return
        bucket.append((key, value))
        self._size += 1
        if self._size / self._cap > 0.75:   # load factor threshold
            self._resize(self._cap * 2)

    def get(self, key, default=None):
        for k, v in self._buckets[self._index(key)]:
            if k == key:
                return v
        return default

    def delete(self, key) -> bool:
        bucket = self._buckets[self._index(key)]
        for i, (k, _) in enumerate(bucket):
            if k == key:
                bucket.pop(i)
                self._size -= 1
                return True
        return False

    def __len__(self) -> int:
        return self._size

    def __contains__(self, key) -> bool:
        return self.get(key, _MISSING) is not _MISSING


_MISSING = object()

---
## Real-World DSA Scenarios — ShopFlow, BuildFast, RideStream
